Q5] Writing Viterbi Algorithm for the Primer 

### Hidden Markov Model (HMM) Configuration

---

#### **States**
- `E`: Exon  
- `5`: 5′ Splice Site  
- `I`: Intron  

---

#### **Alphabet (Bases)**
`{ A, C, G, T }`

---

#### **Transition Probabilities**

| From   | To   | Probability |
|--------|------|-------------|
| Start  | E    | 1.0         |
| E      | E    | 0.9         |
| E      | 5    | 0.1         |
| 5      | I    | 1.0         |
| I      | I    | 0.9         |
| I      | End  | 0.1         |

---

#### **Emission Probabilities**

| State | A    | C    | G    | T    |
|-------|------|------|------|------|
| **E** | 0.25 | 0.25 | 0.25 | 0.25 |
| **5** | 0.05 | 0.00 | 0.95 | 0.00 |
| **I** | 0.40 | 0.10 | 0.10 | 0.40 |


In [ ]:
import math

# Log-safe wrapper
def log(x):
    return -math.inf if x == 0 else math.log(x)

# Compute log-probability of a path
def log_prob_path(path: str, seq: str) -> float:
    if len(path) != len(seq):
        raise ValueError("Path and sequence lengths must match")
    
    prob = log(start_p[path[0]]) + log(emit_p[path[0]].get(seq[0], 0))
    for i in range(1, len(seq)):
        prev, curr = path[i-1], path[i]
        prob += log(trans_p[prev].get(curr, 0)) + log(emit_p[curr].get(seq[i], 0))

    if path[-1] == 'I':
        prob += log(trans_p['I'].get('end', 0))
    return prob

# HMM Parameters
states = ['E', '5', 'I']
start_p = {'E': 1.0, '5': 0.0, 'I': 0.0}
trans_p = {
    'E': {'E': 0.9, '5': 0.1},
    '5': {'I': 1.0},
    'I': {'I': 0.9, 'end': 0.1}
}
emit_p = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.0, 'G': 0.95, 'T': 0.0},
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}
}

# Sample input
seq = "CTTCATGTGAAAGCAGACGTAAGTCA"
given_path = "EEEEEEEEEEEEEEEEEE5IIIIIII"

# Output log prob of given path
print("Log prob of given path:", round(log_prob_path(given_path, seq), 2))

Log prob of given path: -41.22


In [ ]:
# Viterbi algorithm
V = [{}]
paths = {}

# Init
for s in states:
    V[0][s] = log(start_p[s]) + log(emit_p[s].get(seq[0], 0))
    paths[s] = [s]

# DP steps
for i in range(1, len(seq)):
    V.append({})
    new_paths = {}

    for curr in states:
        max_p, best_prev = -math.inf, None
        for prev in states:
            tp = trans_p.get(prev, {}).get(curr, 0)
            ep = emit_p[curr].get(seq[i], 0)
            if tp > 0 and ep > 0:
                p = V[i-1][prev] + log(tp) + log(ep)
                if p > max_p:
                    max_p, best_prev = p, prev
        V[i][curr] = max_p
        new_paths[curr] = paths[best_prev] + [curr] if best_prev else [curr]
    paths = new_paths

# Termination
final_idx = len(seq) - 1
final_prob, final_state = max((V[final_idx][s], s) for s in states)
best_path = ''.join(paths[final_state])

print("Viterbi best log Prob:", round(final_prob, 2))
print("Most likely path:", best_path)

Viterbi best log Prob: -38.68
Most likely path: EEEEEEEEEEEEEEEEEEEEEEEEEE
